# Phase 3 — LLMs & RAG
## Day 17: LLM APIs — OpenAI / Anthropic / Groq

**What I'm building:**
- Understand the messages format: system / user / assistant roles
- Control generation: temperature, top_p, max_tokens
- Get structured JSON output from an LLM
- Build a domain Q&A bot with a strong system prompt

**APIs covered:** Groq (free) → Anthropic → OpenAI format

In [1]:
!pip install -q groq anthropic openai

import os
import json
from groq import Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 21.7 MB/s eta 0:00:0000:01


## Step 1: API Keys — The Right Way

API keys are secrets. They never go in source code.


In [2]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")

# Verify it loaded (never print the actual key)
print(f"Key loaded: {'✅' if GROQ_API_KEY else '❌'}")
print(f"Key prefix: {GROQ_API_KEY[:8]}...")

Key loaded: ✅
Key prefix: gsk_i6Kn...


## Step 2: Your First API Call — Anatomy of a Request

The messages array IS the conversation.
- system: shapes all model behaviour
- user: what we ask
- assistant: model's previous replies (for memory)

The model has NO memory. We give it history explicitly.

In [3]:
client = Groq(api_key=GROQ_API_KEY)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": "You are a concise AI tutor. Explain concepts clearly in 3 sentences max."
        },
        {
            "role": "user", 
            "content": "What is a neural network?"
        }
    ],
    temperature=0.7,
    max_tokens=200
)

# The actual text lives here — everything else is metadata
answer = response.choices[0].message.content
print(answer)
print("\n--- Response Metadata ---")
print(f"Model: {response.model}")
print(f"Tokens used — prompt: {response.usage.prompt_tokens}, completion: {response.usage.completion_tokens}")

A neural network is a computer system inspired by the human brain, consisting of interconnected nodes (neurons) that process and transmit information. These nodes receive input, apply complex calculations, and produce output, allowing the network to learn and make predictions or decisions. By training on large datasets, neural networks can recognize patterns and improve their performance over time.

--- Response Metadata ---
Model: llama-3.3-70b-versatile
Tokens used — prompt: 57, completion: 70


## Step 3: Temperature — Seeing the Difference

Same question. Same model. Different temperature.
Watch how the output changes character.

In [4]:
def ask(question, temperature, label):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a creative writer."},
            {"role": "user", "content": question}
        ],
        temperature=temperature,
        max_tokens=100
    )
    print(f"\n{'='*50}")
    print(f"Temperature: {temperature} ({label})")
    print(f"{'='*50}")
    print(response.choices[0].message.content)

question = "Describe what happens inside a neural network in one sentence."

ask(question, temperature=0.0, label="Deterministic")
ask(question, temperature=0.7, label="Balanced")
ask(question, temperature=1.5, label="Creative/Chaotic")


Temperature: 0.0 (Deterministic)
As data flows through a neural network, complex algorithms and intricate webs of interconnected nodes, or "neurons," process and transform the information, layer by layer, allowing the network to learn, recognize patterns, and make predictions or decisions based on the input it receives.

Temperature: 0.7 (Balanced)
As data flows through a neural network, complex algorithms and layered nodes, akin to a labyrinthine city of interconnected neurons, process and transform the information, weighing and combining inputs to produce outputs through a symphony of mathematical calculations and adaptive learning.

Temperature: 1.5 (Creative/Chaotic)
As data flows through a neural network, complex algorithms and interconnected nodes, or "neurons," process and transform the information, amplifying or dampening various signal patterns through layers of weighted calculations, ultimately generating predictions, classifications, or outputs that reflect the network's le

## Step 4: Multi-Turn Conversation — Giving the Model Memory

The model forgets everything between calls.
To create a "conversation", we append each exchange to the messages list
and pass the entire history on every call.

In [5]:
def chat_session():
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert AI/ML tutor. "
                "You explain concepts clearly, use analogies, and give concrete examples. "
                "Keep answers under 5 sentences unless the user asks for more."
            )
        }
    ]
    
    print("AI Tutor — type 'quit' to exit\n")
    
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() == "quit":
            break
        if not user_input:
            continue
            
        # Add user message to history
        messages.append({"role": "user", "content": user_input})
        
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,  # Full history every time
            temperature=0.7,
            max_tokens=300
        )
        
        assistant_reply = response.choices[0].message.content
        
        # Add model reply to history so next turn has context
        messages.append({"role": "assistant", "content": assistant_reply})
        
        print(f"\nTutor: {assistant_reply}\n")
    
    print(f"\nConversation ended. Total exchanges: {(len(messages)-1)//2}")
    return messages

conversation_history = chat_session()

AI Tutor — type 'quit' to exit



You:  What is backpropagation?



Tutor: Backpropagation is an essential algorithm in machine learning that helps neural networks learn from their mistakes. Imagine you're trying to hit a target with an arrow, but you miss - backpropagation is like adjusting your aim based on how far off you were and in which direction. It works by calculating the error between the network's predictions and the actual outputs, then propagating this error backwards through the layers to adjust the weights and biases. This process is repeated until the network's predictions are accurate, allowing it to learn and improve over time.



You:  Can you give me an analogy?



Tutor: Think of backpropagation like a delivery truck trying to reach a specific house. The truck (neural network) takes a route (makes predictions) but ends up at the wrong house (makes an error). The truck then checks the map (calculates the error), figures out where it went wrong (identifies the mistakes), and adjusts its route (adjusts the weights and biases) to try again, eventually finding the correct house (makes accurate predictions).



You:  How does it relate to gradient descent?



Tutor: Backpropagation and gradient descent are closely related: backpropagation is used to calculate the gradient of the error with respect to the model's parameters, and gradient descent is the optimization algorithm that uses this gradient to update the parameters and minimize the error. In other words, backpropagation computes the direction of the update, and gradient descent takes a step in that direction to adjust the parameters.



You:  quit



Conversation ended. Total exchanges: 3


## Step 5: JSON Mode — Structured Output for Real Applications

Text output is for humans. JSON output is for code.
JSON mode forces the model to return valid, parseable JSON.
Critical for: data extraction, classification APIs, any downstream processing.

In [6]:
def analyze_text(text: str) -> dict:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a text analysis API. "
                    "Always respond with valid JSON only. No explanation, no markdown. "
                    "Return exactly this structure:\n"
                    '{"sentiment": "positive|negative|neutral", '
                    '"confidence": 0.0-1.0, '
                    '"key_topics": ["topic1", "topic2"], '
                    '"summary": "one sentence summary"}'
                )
            },
            {"role": "user", "content": f"Analyze this text: {text}"}
        ],
        temperature=0.0,  # Deterministic for structured output
        max_tokens=200,
        response_format={"type": "json_object"}  # JSON mode
    )
    
    raw = response.choices[0].message.content
    return json.loads(raw)  # Parse string → Python dict

# Test it
texts = [
    "I just deployed my first RAG chatbot and it's working perfectly! The retrieval accuracy is amazing.",
    "The model keeps hallucinating facts that aren't in my documents. Very frustrating.",
    "Transfer learning involves using pretrained weights as a starting point for a new task."
]

for text in texts:
    result = analyze_text(text)
    print(f"\nText: {text[:60]}...")
    print(f"Sentiment: {result['sentiment']} (confidence: {result['confidence']})")
    print(f"Topics: {result['key_topics']}")
    print(f"Summary: {result['summary']}")


Text: I just deployed my first RAG chatbot and it's working perfec...
Sentiment: positive (confidence: 0.9)
Topics: ['RAG chatbot', 'retrieval accuracy']
Summary: The user successfully deployed their first RAG chatbot with impressive retrieval accuracy.

Text: The model keeps hallucinating facts that aren't in my docume...
Sentiment: negative (confidence: 0.9)
Topics: ['model performance', 'frustration']
Summary: The model is producing inaccurate facts, causing frustration.

Text: Transfer learning involves using pretrained weights as a sta...
Sentiment: neutral (confidence: 0.8)
Topics: ['transfer learning', 'pretrained weights']
Summary: Transfer learning uses pretrained weights as a starting point for new tasks.


## Step 6: Domain Q&A Bot — Putting It Together

A system prompt that makes the model behave like a specialized assistant.
Key elements of a strong system prompt:
1. Role definition (who the model IS)
2. Constraints (what it must/must not do)  
3. Output format (how it should respond)
4. Fallback behaviour (what to do when it doesn't know)

In [7]:
AI_TUTOR_SYSTEM_PROMPT = """You are Nexus, an expert AI/ML interview coach for junior AI engineers.

Your expertise covers:
- Deep Learning (PyTorch, CNNs, LSTMs, transformers)
- NLP & HuggingFace (BERT, fine-tuning, embeddings)
- LLMs & RAG (vector databases, retrieval, agents)
- Deployment (FastAPI, Docker, HuggingFace Spaces)

Rules you follow without exception:
- Answer only AI/ML questions. For anything else, say: "I'm specialized in AI/ML — ask me anything in that domain."
- Always give a concrete example after every explanation.
- If someone asks about your projects, refer them to: github.com/faisalimam1
- End every answer with one follow-up question to deepen understanding.
- Never say "I don't know" — instead say "Let me break down what I do know about this..."

Format: Explanation → Example → Follow-up question."""

class AITutorBot:
    def __init__(self):
        self.messages = [{"role": "system", "content": AI_TUTOR_SYSTEM_PROMPT}]
        self.client = Groq(api_key=GROQ_API_KEY)
    
    def ask(self, question: str) -> str:
        self.messages.append({"role": "user", "content": question})
        
        response = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=self.messages,
            temperature=0.7,
            max_tokens=500
        )
        
        reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": reply})
        return reply
    
    def reset(self):
        self.messages = [{"role": "system", "content": AI_TUTOR_SYSTEM_PROMPT}]
        print("Conversation reset.")

bot = AITutorBot()

# Test 1: On-topic question
print("Q: What is RAG and why is it better than fine-tuning?")
print(bot.ask("What is RAG and why is it better than fine-tuning?"))

print("\n" + "="*60 + "\n")

# Test 2: Off-topic (should hit the constraint)
print("Q: What's the best recipe for biryani?")
print(bot.ask("What's the best recipe for biryani?"))

print("\n" + "="*60 + "\n")

# Test 3: Follow the bot's own follow-up question from Test 1
print("Q: Follow up on RAG")
print(bot.ask("When would fine-tuning actually be better than RAG?"))

Q: What is RAG and why is it better than fine-tuning?
RAG (Retrieval-Augmented Generation) is a paradigm that combines the strengths of retrieval-based and generation-based approaches in natural language processing. It involves using a retriever to fetch relevant information from a database or knowledge base and then using a generator to create a response based on the retrieved information. This approach is particularly useful for tasks that require generating text based on specific knowledge or context.

In contrast to fine-tuning, which involves adjusting the weights of a pre-trained model to fit a specific task, RAG provides a more modular and flexible approach. Fine-tuning can be limited by the amount of training data available and can result in overfitting, whereas RAG can leverage large amounts of external knowledge and adapt to new tasks more easily.

For example, consider a question-answering task where the goal is to answer user questions based on a large corpus of text. A RAG

## Day 17 Summary

**What I built:**
- Understood the messages format: system / user / assistant
- Controlled generation with temperature, top_p, max_tokens  
- Built a JSON extraction API using structured output mode
- Built a domain Q&A bot with a constrained system prompt

**Key insight:** The system prompt is the most powerful tool in LLM engineering.
Every RAG pipeline, every agent, every chatbot is built on this messages array.

**Tomorrow (Day 18):** Prompt Engineering — zero-shot, few-shot, chain-of-thought, ReAct, prompt injection.